In [13]:
pip install scikit-learn pandas numpy joblib

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [15]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [16]:
df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# 'customerID' is useless for machine learning predictions, so let's drop it
df = df.drop(columns=['customerID'])

# Convert our target variable 'Churn' from text (Yes/No) to numbers (1/0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [17]:
X = df.drop(columns=['Churn'])
y = df['Churn']

# Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)



In [18]:
# 4. Identify Feature Types Automatically
# Numeric features need scaling. Categorical features need encoding.
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in X.columns if col not in numeric_features]


In [19]:
# 5. Define the Preprocessing Pipeline
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first')) # drop='first' prevents multicollinearity
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [20]:
# 6. Create the Full Pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [21]:
# 7. Hyperparameter Tuning using GridSearchCV
# These are the settings the grid search will automatically experiment with
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10, None],
    'classifier__min_samples_split': [2, 5]
}

print("Starting Grid Search Tuning... (This might take a moment)")
grid_search = GridSearchCV(full_pipeline, param_grid, cv=3, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best settings found: {grid_search.best_params_}\n")


Starting Grid Search Tuning... (This might take a moment)
Best settings found: {'classifier__max_depth': 10, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}



In [22]:
# 8. Evaluate the Tuned Model
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)

print("--- Evaluation Results ---")
print("Accuracy Score:", accuracy_score(y_test, predictions))
print("\nClassification Report:\n", classification_report(y_test, predictions))


--- Evaluation Results ---
Accuracy Score: 0.8019872249822569

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.79      0.80      0.79      1409



In [23]:
# 9. Export the Pipeline
joblib.dump(best_model, 'telco_churn_production_pipeline.pkl')
print("Pipeline saved completely as 'telco_churn_production_pipeline.pkl'!")

Pipeline saved completely as 'telco_churn_production_pipeline.pkl'!
